# Exploratory analysis: diabetes readmission
This notebook reads the cleaned encounter table from the project pipeline. Interpret rates as encounter-level associations, not causal effects.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
df = pd.read_csv(ROOT / 'data/processed/cleaned_encounters.csv')
df.shape, df.patient_nbr.nunique(), df.readmit_30d.mean()

## Data quality and outcomes
`?` is treated as missing. The pipeline removes full duplicate rows and only excludes target labels other than `<30`, `>30`, and `NO`. See `data_quality.json` for exact counts.

In [ ]:
df.dtypes.to_frame('dtype').join(df.isna().sum().rename('missing')).sort_values('missing', ascending=False).head(15)

In [ ]:
df.readmitted.value_counts(dropna=False), df.groupby('readmitted').readmit_30d.agg(['size','mean'])

## Encounter patterns
Rates are unadjusted; group sizes should be checked alongside the rate. The reports/figures directory contains saved pipeline charts.

In [ ]:
for col in ['age_group','stay_group','prior_inpatient_group','medication_change','admission_type_id']:
    display(df.groupby(col, dropna=False).readmit_30d.agg(encounters='size', early_readmissions='sum', rate='mean').reset_index())

In [ ]:
fig, ax = plt.subplots(figsize=(9,4))
df.groupby('age_group').readmit_30d.mean().plot.bar(ax=ax)
ax.set(title='Early readmission rate by age group', xlabel='Age group', ylabel='Encounter rate')
plt.tight_layout()

## Sensitivity and limitations
The primary cohort retains death/hospice disposition rows. Compare the sensitivity summary in `reports/eda_summary.json` or execute the following. Associations can reflect severity, practice patterns, access, or coding and are not evidence of causation.

In [ ]:
df.groupby('death_hospice_disposition').readmit_30d.agg(encounters='size', rate='mean')